# Create a connection

In [ ]:
import requests
import sqlite3

open("company_operations.db", "wb") \
    .write(requests.get("https://github.com/thomasnield/anaconda_intro_to_sql/raw/refs/heads/main/company_operations.db").content)

conn = sqlite3.conn("company_operations.db")

# Something wrong with SQL syntax

In [ ]:
import pandas as pd

pd.read_sql("SELECT ALL FROM CUSTOMER", conn)

# What in the world?

In [ ]:
sql = """
SELECT 'ORDER_DATE'
FROM CUSTOMER_ORDER
"""

pd.read_sql(sql, conn)


# Was this SQLite or SQL Server?

In [ ]:
import pandas as pd

pd.read_sql("SELECT TOP 1 * FROM CUSTOMER", conn)

# Why is my date formatting all messed up?

In [ ]:
sql = """
SELECT CUSTOMER_ORDER_ID, ORDER_DATE, strftime('%d/%M/%Y', ORDER_DATE) AS FORMATTED_DATE
FROM CUSTOMER_ORDER
"""

pd.read_sql(sql, conn)


# Where is the world is Alpha Medical?

In [ ]:
sql = """
SELECT
    CUSTOMER_ORDER_ID,
    CUSTOMER.CUSTOMER_ID,
    CUSTOMER_NAME,
    ADDRESS,
    CITY,
    STATE,
    ZIP,
    ORDER_DATE,
    PRODUCT_ID,
    QUANTITY
FROM CUSTOMER INNER JOIN CUSTOMER_ORDER
ON CUSTOMER.CUSTOMER_ID = CUSTOMER_ORDER.CUSTOMER_ID

WHERE CUSTOMER_NAME = 'Alpha Medical'

ORDER BY CUSTOMER.CUSTOMER_ID
"""

pd.read_sql(sql, conn)

# Let's bring in `PRODUCT`. Wait, why did Alpha Medical disappear again?

In [ ]:
sql = """
SELECT
CUSTOMER_ORDER_ID,
CUSTOMER.CUSTOMER_ID,
CUSTOMER_NAME,
ADDRESS,
CITY,
STATE,
ZIP,
ORDER_DATE,
PRODUCT.PRODUCT_ID,
QUANTITY,
PRICE

FROM CUSTOMER LEFT JOIN CUSTOMER_ORDER
ON CUSTOMER.CUSTOMER_ID = CUSTOMER_ORDER.CUSTOMER_ID

INNER JOIN PRODUCT
ON PRODUCT.PRODUCT_ID = CUSTOMER_ORDER.PRODUCT_ID

WHERE CUSTOMER_NAME = 'Alpha Medical'

ORDER BY CUSTOMER.CUSTOMER_ID
"""

pd.read_sql(sql, conn)

# Why is this not giving me yesterday's date?

In [ ]:
sql = """
SELECT DATE('now') - 1 AS YESTERDAY
"""

pd.read_sql(sql, conn)

# Somebody told me I should use common table expressions. What's that?

In [15]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("company_operations.db")

sql = """
SELECT
CUSTOMER_ID,
ORDER_DATE,
QUANTITY,
(SELECT AVG(QUANTITY)
 FROM CUSTOMER_ORDER co3
 WHERE co3.CUSTOMER_ID = co1.CUSTOMER_ID) as avg_customer_quantity
FROM CUSTOMER_ORDER co1
ORDER BY ORDER_DATE
"""

pd.read_sql(sql, conn)

Query took 0.0463 seconds


# They also said I should learn to avoid unncessary joins

In [26]:
sql = """

SELECT
    COALESCE(t.YEAR, nt.YEAR) AS YEAR,
    COALESCE(t.MONTH, nt.MONTH) AS MONTH,
    COALESCE(t.TOTAL_TORNADO_RAIN, 0) AS TOTAL_TORNADO_RAIN,
    COALESCE(nt.TOTAL_NON_TORNADO_RAIN, 0) AS TOTAL_NON_TORNADO_RAIN
FROM (
        SELECT
        CAST(strftime('%Y', REPORT_DATE) AS INTEGER) AS YEAR,
        CAST(strftime('%m', REPORT_DATE) AS INTEGER) AS MONTH,
        SUM(RAIN) AS TOTAL_TORNADO_RAIN
        FROM WEATHER_MONITOR
        WHERE TORNADO = 1
        GROUP BY YEAR, MONTH
    ) AS t
FULL OUTER JOIN (
    SELECT
        CAST(strftime('%Y', REPORT_DATE) AS INTEGER) AS YEAR,
        CAST(strftime('%m', REPORT_DATE) AS INTEGER) AS MONTH,
        SUM(RAIN) AS TOTAL_NON_TORNADO_RAIN
    FROM WEATHER_MONITOR
    WHERE TORNADO = 0
    GROUP BY YEAR, MONTH
) AS nt

ON t.YEAR = nt.YEAR AND t.MONTH = nt.MONTH;
"""

pd.read_sql(sql, conn)


,YEAR,MONTH,TOTAL_TORNADO_RAIN,TOTAL_NON_TORNADO_RAIN
0,2021,2,15.22,122.85
1,2021,3,24.92,104.11
2,2021,4,9.87,143.92
3,2021,5,19.88,138.36
4,2020,11,0.00,392.22
5,2020,12,0.00,433.16
6,2021,1,0.00,316.27
